In [7]:
!pip install matplotlib

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 5.7 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.1 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.1 MB 16.6 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 15.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 55.3 MB/s  0:00:00
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ---------------------------- ----------- 5.0/7.0 MB 31.4 MB/s eta 0:00:01
   -------------------------------------- - 6.8/7.0 MB 17.6 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 15.4 MB/s  0:00:00

   ---------------------------------------- 0/7 [pyparsing]
   ---------------------------------------- 0/7 [pyparsing]
   ----- ---------------------------------- 1/7 [pillow]
   ----- -

# Carregando a base de dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
ibovespa = pd.read_csv(r"../data/raw/Dados Históricos - Ibovespa.csv", sep=',')
ibovespa.head()

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
0,19.02.2026,188.534,186.020,188.687,185.928,"9,24B","1,35%"
1,18.02.2026,186.016,186.464,187.657,185.001,"7,79B","-0,24%"
2,13.02.2026,186.464,187.766,187.766,183.662,"11,59B","-0,69%"
3,12.02.2026,187.766,189.694,189.990,186.959,"12,36B","-1,02%"
4,11.02.2026,189.699,185.936,190.561,185.936,"11,64B","2,03%"


In [5]:
# Quais os tipos informações compõe o banco de dados (base original)
ibovespa.info()

<class 'pandas.DataFrame'>
RangeIndex: 534 entries, 0 to 533
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Data      534 non-null    str    
 1   Último    534 non-null    float64
 2   Abertura  534 non-null    float64
 3   Máxima    534 non-null    float64
 4   Mínima    534 non-null    float64
 5   Vol.      534 non-null    str    
 6   Var%      534 non-null    str    
dtypes: float64(4), str(3)
memory usage: 29.3 KB


# EDA - Análise Exploratória de Dados

In [6]:
# renomeando as colunas
ibovespa.rename(columns={'Data': 'data', 'Abertura': 'abertura', 'Máxima': 'maxima', 'Mínima': 'minima', 'Último': 'fechamento', 'Vol.': 'volume', "Var%": "variacao_percentual"}, inplace=True)
ibovespa.head()

,data,fechamento,abertura,maxima,minima,volume,variacao_percentual
0,19.02.2026,188.534,186.020,188.687,185.928,"9,24B","1,35%"
1,18.02.2026,186.016,186.464,187.657,185.001,"7,79B","-0,24%"
2,13.02.2026,186.464,187.766,187.766,183.662,"11,59B","-0,69%"
3,12.02.2026,187.766,189.694,189.990,186.959,"12,36B","-1,02%"
4,11.02.2026,189.699,185.936,190.561,185.936,"11,64B","2,03%"


In [87]:
# Transformando a coluna Data de objeto para data
ibovespa["data"] = pd.to_datetime(ibovespa["data"])
print(ibovespa.head())

        data  fechamento  abertura   maxima   minima  volume   
0 2026-02-19     188.534   186.020  188.687  185.928   9,24B  \
1 2026-02-18     186.016   186.464  187.657  185.001   7,79B   
2 2026-02-13     186.464   187.766  187.766  183.662  11,59B   
3 2026-02-12     187.766   189.694  189.990  186.959  12,36B   
4 2026-02-11     189.699   185.936  190.561  185.936  11,64B   

  variacao_percentual  
0               1,35%  
1              -0,24%  
2              -0,69%  
3              -1,02%  
4               2,03%  


/var/folders/x7/hy6kgv1x6_s5fx88qm5c36t40000gn/T/ipykernel_44543/1809113291.py:2: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  ibovespa["data"] = pd.to_datetime(ibovespa["data"])


In [88]:
# Ajustando a coluna de variação
ibovespa["variacao_percentual"] = ibovespa["variacao_percentual"].str.replace('%', '').str.replace(',', '.').astype(float)
print(ibovespa.head())

        data  fechamento  abertura   maxima   minima  volume   
0 2026-02-19     188.534   186.020  188.687  185.928   9,24B  \
1 2026-02-18     186.016   186.464  187.657  185.001   7,79B   
2 2026-02-13     186.464   187.766  187.766  183.662  11,59B   
3 2026-02-12     187.766   189.694  189.990  186.959  12,36B   
4 2026-02-11     189.699   185.936  190.561  185.936  11,64B   

   variacao_percentual  
0                 1.35  
1                -0.24  
2                -0.69  
3                -1.02  
4                 2.03  


In [89]:
# Quantidade de dados nulos por coluna
ibovespa.isnull().sum()

data                   0
fechamento             0
abertura               0
maxima                 0
minima                 0
volume                 0
variacao_percentual    0
dtype: int64

In [90]:
# Criando uma coluna se fechou em alta ou em baixa
ibovespa['status'] = ibovespa.apply(lambda row: 'alta' if row['variacao_percentual'] > 0 else 'baixa', axis=1)
ibovespa.head()

,data,fechamento,abertura,maxima,minima,volume,variacao_percentual,status
0,2026-02-19,188.534,186.020,188.687,185.928,"9,24B",1.35,alta
1,2026-02-18,186.016,186.464,187.657,185.001,"7,79B",-0.24,baixa
2,2026-02-13,186.464,187.766,187.766,183.662,"11,59B",-0.69,baixa
3,2026-02-12,187.766,189.694,189.990,186.959,"12,36B",-1.02,baixa
4,2026-02-11,189.699,185.936,190.561,185.936,"11,64B",2.03,alta


In [91]:
# Validando a saída final da base
ibovespa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534 entries, 0 to 533
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   data                 534 non-null    datetime64[ns]
 1   fechamento           534 non-null    float64       
 2   abertura             534 non-null    float64       
 3   maxima               534 non-null    float64       
 4   minima               534 non-null    float64       
 5   volume               534 non-null    object        
 6   variacao_percentual  534 non-null    float64       
 7   status               534 non-null    object        
dtypes: datetime64[ns](1), float64(5), object(2)
memory usage: 33.5+ KB


In [92]:
# Considerando apenas as colunas relevantes
ibovespa_final = ibovespa[['data', 'abertura', 'maxima', 'minima', 'fechamento', 'variacao_percentual', 'status']]

In [93]:
# Salvando dataframe final em csv
ibovespa_final.to_csv('../data/trusted/ibovespa_historico.csv', index=False, sep = ';')